# Transformer Decoder

## Code

In [10]:
import torch
import torch.nn as nn

import models.deep_learning.architectures as mynn


## Testing

In [11]:
# input parameters
N = 3
batch_size = 2
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.float32

# Encoder Layer parameters
d_model = 4
nhead = 2
dim_feedforward = 64
dropout = 0.2
layer_norm_eps = 1e-5
batch_first = True
norm_first = True
bias = True

## Transformer encoder parameters
num_layers = 3
#  It helps only when norm_first is True,
norm = None  # nn.LayerNorm(d_model).to(device=device,dtype=dtype)

In [3]:
torch.manual_seed(0)
x = torch.randn(batch_size, N, d_model, device=device, dtype=dtype)

In [4]:
init_seed = 42  # avoide weights initialization randomness effects
train_seed = 24  # avoid dropout randomness effects

In [ ]:
torch.manual_seed(init_seed)
tf_encl = mynn.TransformerEncoderLayer(
    d_model,
    nhead,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation_cls=nn.GELU,
    layer_norm_eps=layer_norm_eps,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
)
tf_enc = mynn.TransformerEncoder(tf_encl, num_layers, norm=norm)

torch.manual_seed(init_seed)
nn_tf_encl = nn.TransformerEncoderLayer(
    d_model,
    nhead,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation=nn.GELU(),
    layer_norm_eps=layer_norm_eps,
    batch_first=batch_first,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
)
nn_tf_enc = nn.TransformerEncoder(
    nn_tf_encl, num_layers, norm=norm, enable_nested_tensor=False
)


### Evaluation

In [6]:
tf_enc.eval()
tf_enc(x)

tensor([[[-0.0296, -0.0916,  1.0885, -1.0260],
         [ 0.6635,  0.4575,  1.4182, -0.2418],
         [ 0.4502,  0.3450,  0.7149,  0.3737]],

        [[ 0.1837, -0.0224,  1.0295,  1.6342],
         [ 0.3983, -0.1241,  0.9514,  0.5204],
         [ 1.1837,  1.0843,  1.0421,  0.6918]]], device='mps:0',
       grad_fn=<AddBackward0>)

In [7]:
nn_tf_enc.eval()
nn_tf_enc(x)

tensor([[[-0.0296, -0.0916,  1.0885, -1.0260],
         [ 0.6635,  0.4575,  1.4182, -0.2418],
         [ 0.4502,  0.3450,  0.7149,  0.3737]],

        [[ 0.1837, -0.0224,  1.0295,  1.6342],
         [ 0.3983, -0.1241,  0.9514,  0.5204],
         [ 1.1837,  1.0843,  1.0421,  0.6918]]], device='mps:0',
       grad_fn=<AddBackward0>)

### Training

In [8]:
mse = torch.nn.MSELoss()

In [9]:
torch.manual_seed(train_seed)
nn_tf_enc.train()
print(nn_tf_enc(x))
loss = mse(nn_tf_enc(x), x)
print(loss.item())
loss.backward()
nn_tf_enc(x)


tensor([[[ 0.6852, -0.1630,  2.2074, -0.9302],
         [ 0.9349,  0.6778,  1.9027, -0.1390],
         [ 0.8390,  0.4592,  1.5378,  0.0823]],

        [[-0.1834,  0.3312,  0.7525,  2.1237],
         [ 0.7906,  0.1192,  1.3922,  0.4578],
         [ 0.8707,  0.1922,  0.5967,  0.7011]]], device='mps:0',
       grad_fn=<AddBackward0>)
0.6275849342346191


tensor([[[ 0.2741, -0.4894,  1.8790, -1.0957],
         [ 0.7022,  0.7676,  0.8320, -0.2268],
         [ 0.7146,  0.2815,  0.8392,  1.0298]],

        [[ 0.5085,  0.0161,  1.4960,  2.1186],
         [ 0.4879,  0.0038,  1.0066,  1.0014],
         [ 1.1819,  0.7470,  0.2558,  0.4909]]], device='mps:0',
       grad_fn=<AddBackward0>)

In [10]:
torch.manual_seed(train_seed)
tf_enc.train()
print(tf_enc(x))
loss = mse(tf_enc(x), x)
print(loss.item())
loss.backward()
tf_enc(x)

tensor([[[ 0.6852, -0.1630,  2.2074, -0.9302],
         [ 0.9349,  0.6778,  1.9027, -0.1390],
         [ 0.8390,  0.4592,  1.5378,  0.0823]],

        [[-0.1834,  0.3312,  0.7525,  2.1237],
         [ 0.7906,  0.1192,  1.3922,  0.4578],
         [ 0.8707,  0.1922,  0.5967,  0.7011]]], device='mps:0',
       grad_fn=<AddBackward0>)
0.6275849342346191


tensor([[[ 0.2741, -0.4894,  1.8790, -1.0957],
         [ 0.7022,  0.7676,  0.8320, -0.2268],
         [ 0.7146,  0.2815,  0.8392,  1.0298]],

        [[ 0.5085,  0.0161,  1.4960,  2.1186],
         [ 0.4879,  0.0038,  1.0066,  1.0014],
         [ 1.1819,  0.7470,  0.2558,  0.4909]]], device='mps:0',
       grad_fn=<AddBackward0>)